# 12 — Pós-teste: O/E, ECE, curva de decisão e equidade

Lê **somente as previsões já gravadas pelo notebook 11** (`outputs/predictions/*_predicoes_teste.parquet`). **Não chama** `load_supervised_artifact` com blocos de teste e **não reabre o teste**: o G9 já está `executado` e este notebook não precisa (nem pode) alterá-lo.

Produz os números citados em `docs/decisoes.md` seção 9 e no `README.md` que, até aqui, não tinham código reprodutível no repositório (ver Auditoria_repo_limpo.md, item L5-02): O/E, inclinação de calibração e ECE por janela e por ano; curva de decisão (benefício líquido); e tabelas de desempenho/calibração por região, faixa etária e UF (essas três colunas já vêm nas previsões gravadas). Equidade por sexo/raça/zona exigiria um join adicional com a coorte publicada por `id_registro` — fica como próximo passo, não incluído aqui para manter este notebook simples e conferível.

In [ ]:
from pathlib import Path
import sys

ROOT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "supervised.yaml").exists())
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression

from src import config as C
from src.avaliacao import ece_quantis, mce_quantis, brier, metricas_por_grupo

C.carregar_dotenv()
C.garantir_diretorios()
print("Raiz:", ROOT_DIR)

In [ ]:
import json

TABLES_DIR = C.OUTPUTS_DIR / "tables"
PRED_DIR = C.OUTPUTS_DIR / "predictions"

manifestos = sorted(TABLES_DIR.glob("11_validacao_final_*_manifesto.json"))
if not manifestos:
    raise FileNotFoundError(
        f"Nenhum manifesto 11_validacao_final_*_manifesto.json em {TABLES_DIR}. "
        "Este notebook so funciona depois do notebook 11 ter aberto o teste."
    )

manifesto_11 = json.loads(manifestos[-1].read_text(encoding="utf-8"))
if not manifesto_11.get("test_opened"):
    raise RuntimeError(f"{manifestos[-1].name} nao tem test_opened=true; nada para ler aqui.")

caminho_predicoes = Path(manifesto_11["artefatos"]["predicoes_teste"])
if not caminho_predicoes.is_absolute():
    caminho_predicoes = ROOT_DIR / caminho_predicoes
if not caminho_predicoes.exists():
    # caminho gravado pode ser de outra maquina (<REPO>\...); tenta pelo nome na pasta padrao
    caminho_predicoes = PRED_DIR / caminho_predicoes.name

pred = pd.read_parquet(caminho_predicoes)
limiar = float(manifesto_11["limiar_final"])
print("Previsoes:", caminho_predicoes, "| linhas:", len(pred), "| limiar:", limiar)
print("Blocos presentes:", sorted(pred["bloco"].unique()))
display(pred.head())

In [ ]:
def inclinacao_calibracao(y_true, y_prob, eps: float = 1e-6):
    """Inclinacao (slope) de Cox: regressao logistica de y_true no logit(y_prob)."""
    p = np.clip(np.asarray(y_prob, dtype=float), eps, 1 - eps)
    logit = np.log(p / (1 - p)).reshape(-1, 1)
    modelo = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
    modelo.fit(logit, np.asarray(y_true, dtype=int))
    return float(modelo.coef_[0][0])


def resumo_janela(y_true, y_prob):
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(y_prob, dtype=float)
    soma_p = float(p.sum())
    return {
        "n": int(len(y)), "positivos": int(y.sum()), "prevalencia": float(y.mean()),
        "o_e": (float(y.sum()) / soma_p) if soma_p > 0 else float("nan"),
        "inclinacao": inclinacao_calibracao(y, p),
        "ece_quantis": float(ece_quantis(y, p, n_bins=10)),
        "mce_quantis": float(mce_quantis(y, p, n_bins=10)),
        "brier": float(brier(y, p)),
    }


JANELAS = {
    "teste_a": pred["bloco"] == "test_ref",
    "pandemico_isolado": pred["bloco"] == "test_pandemic",
    "teste_b": pred["bloco"].isin(["test_ref", "test_pandemic"]),
}

linhas_janela = []
for nome, mascara in JANELAS.items():
    sub = pred.loc[mascara]
    if sub.empty:
        continue
    linhas_janela.append({"janela": nome, **resumo_janela(sub["y_true"], sub["y_prob"])})

calibracao_por_janela = pd.DataFrame(linhas_janela)
display(calibracao_por_janela.round(4))

In [ ]:
if "ano_internacao" not in pred.columns:
    raise KeyError("'ano_internacao' nao esta nas previsoes gravadas pelo notebook 11.")

linhas_ano = []
for ano, sub in pred.groupby("ano_internacao", observed=True):
    if len(sub) == 0:
        continue
    linhas_ano.append({"ano": int(ano), **resumo_janela(sub["y_true"], sub["y_prob"])})

calibracao_por_ano = pd.DataFrame(linhas_ano).sort_values("ano").reset_index(drop=True)
display(calibracao_por_ano.round(4))

In [ ]:
def curva_decisao(y_true, y_prob, limiares):
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(y_prob, dtype=float)
    prevalencia = float(y.mean())
    linhas = []
    for t in limiares:
        alerta = (p >= t).astype(int)
        vp = int(((alerta == 1) & (y == 1)).sum())
        fp = int(((alerta == 1) & (y == 0)).sum())
        peso = t / (1 - t) if t < 1 else float("inf")
        beneficio_modelo = vp / len(y) - (fp / len(y)) * peso
        beneficio_todos = prevalencia - (1 - prevalencia) * peso
        linhas.append({
            "limiar": t, "alertas_por_100": 100 * alerta.mean(),
            "beneficio_modelo": beneficio_modelo, "beneficio_alertar_todos": beneficio_todos,
        })
    return pd.DataFrame(linhas)


limiares_grade = np.round(np.arange(0.02, 0.41, 0.02), 2)
curva_teste_a = curva_decisao(pred.loc[JANELAS["teste_a"], "y_true"], pred.loc[JANELAS["teste_a"], "y_prob"], limiares_grade)
curva_pandemia = curva_decisao(pred.loc[JANELAS["pandemico_isolado"], "y_true"], pred.loc[JANELAS["pandemico_isolado"], "y_prob"], limiares_grade)

fig, eixos = plt.subplots(1, 2, figsize=(11, 4))
for eixo, curva, titulo in ((eixos[0], curva_teste_a, "Teste A"), (eixos[1], curva_pandemia, "2020-2022")):
    eixo.plot(curva["limiar"], curva["beneficio_modelo"], label="modelo")
    eixo.plot(curva["limiar"], curva["beneficio_alertar_todos"], label="alertar todos", linestyle="--")
    eixo.axhline(0, color="grey", linewidth=0.8, label="alertar ninguem")
    eixo.set_title(titulo); eixo.set_xlabel("limiar"); eixo.set_ylabel("beneficio liquido"); eixo.legend()
fig.tight_layout()
CAMINHO_FIGURA_DECISAO = C.FIGURES_DIR / "curva_decisao.png"
fig.savefig(CAMINHO_FIGURA_DECISAO, dpi=150)
print("Figura salva em:", CAMINHO_FIGURA_DECISAO)

In [ ]:
teste_a = pred.loc[JANELAS["teste_a"]].reset_index(drop=True)

equidade_regiao = metricas_por_grupo(
    teste_a, teste_a["y_true"], teste_a["y_prob"], coluna_grupo="regiao_residencia", limiar=limiar, min_n=100,
)
equidade_faixa_etaria = metricas_por_grupo(
    teste_a, teste_a["y_true"], teste_a["y_prob"], coluna_grupo="faixa_etaria", limiar=limiar, min_n=100,
)
display(equidade_regiao.round(4))
display(equidade_faixa_etaria.round(4))

fig2, eixo2 = plt.subplots(figsize=(6, 4))
eixo2.bar(equidade_regiao["grupo"], equidade_regiao["recall"])
eixo2.set_ylabel("sensibilidade"); eixo2.set_title("Sensibilidade por região (Teste A)")
fig2.tight_layout()
CAMINHO_FIGURA_EQUIDADE = C.FIGURES_DIR / "equidade_regiao.png"
fig2.savefig(CAMINHO_FIGURA_EQUIDADE, dpi=150)
print("Figura salva em:", CAMINHO_FIGURA_EQUIDADE)

In [ ]:
FINAL_DIR = C.OUTPUTS_DIR / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

calibracao_por_janela.to_csv(FINAL_DIR / "calibracao_por_janela.csv", index=False)
calibracao_por_ano.to_csv(FINAL_DIR / "calibracao_por_ano.csv", index=False)
curva_teste_a.to_csv(FINAL_DIR / "curva_decisao.csv", index=False)
curva_pandemia.to_csv(FINAL_DIR / "curva_decisao_pandemia.csv", index=False)
equidade_regiao.to_csv(FINAL_DIR / "tabelas_relatorio_regiao_residencia.csv", index=False)
equidade_faixa_etaria.to_csv(FINAL_DIR / "tabelas_relatorio_faixa_etaria.csv", index=False)

print("Tabelas gravadas em:", FINAL_DIR)
for p in sorted(FINAL_DIR.glob("*.csv")):
    print(" -", p.name)